# Khôi phục kết quả D — đúng Output116 — CPU, không train lại
**Accelerator=None · Internet ON · Save & Run All. Không GPU, không load mô hình.**
Add Input duy nhất: relation-kv-preparation-v115. Bật Secret KAGGLE_API_TOKEN;
nếu secret tên codex thì sửa SECRET_NAME. Không dán token vào code/chat.
Tải đúng version116, gồm checkpoint chỉ để kiểm SHA (không torch.load).
Kiểm đủ2worker ENDPOINT_500_COMPLETE, captions1000/held99, cached CIDEr/CHAIR và metadata.
Giữ nguyên file nguồn, tạo thư mục recovery mới; không gọi prepare/worker/training.
Nếu thiếu artifact thì dừng và xuất ZIP/log, không tự dùng version khác hoặc train lại.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/116'
OWNER='ducanh2403';SLUG='unused-recovery';TITLE='CPU summary recovery'
SECRET_NAME='KAGGLE_API_TOKEN'
PREPARED_ROOT=Path('/kaggle/input/datasets/ducanh2403/relation-kv-preparation-v115')
OUTPUT=Path('/kaggle/working/relation_kv_D_recovered_v116')
REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
if OUTPUT.exists() or REVIEW.exists():raise FileExistsError('Preserve old recovery; use a fresh session/path')
if not (PREPARED_ROOT/'metadata.json').is_file():raise FileNotFoundError('Add Input relation-kv-preparation-v115')
print('CPU recovery only. Training/model/GPU will NOT run.',flush=True)


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/116':raise ValueError('This notebook pins116; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/116', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v116', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v116; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import tempfile,hashlib,json,urllib.request,zipfile,traceback,os
CORE='744264b55496935ad5768536c8d19ec1601d424d'
ASSETS={'experiments/semantic_grounding_method/kv_data.py': '37cd1501237cf43a0e919f437b7ea046c824a6db7a8e2748084d93d1fc207901', 'experiments/semantic_grounding_method/kv_pilot_data.py': '8a7c747495729b5829b99a441c5c93e29d45c07563a598880e3183b469de1528', 'experiments/semantic_grounding_method/kv_pilot_evaluate.py': '6386de387919ba696bdc19b5ecbc46ea75e4b61396ae8e79851c3d47e312c734', 'experiments/semantic_grounding_method/recover_kv_summary_cpu.py': 'a6e161d72cedd0c39b3e143d850a2235acaaaa55f0103720ea180b8961fbf62b', 'experiments/semantic_grounding_method/test_kv_json_regression.py': '9b5bd4ae8ca6edeacfc15f1c80f98fb992526f36f8f2d5eca3411a9a533c9462', 'experiments/semantic_grounding_method/KV_SAVE_V115_PACKAGE.json': '82521c3949d6129d024620aef6fe9e35f3d1074c4051d05a54a0db1ff712c964', 'experiments/relation_frames_inventory/inventory.py': '8975c6a221e3d769c5449901347717f8776c4ae191fb4fad5d3eb117fe18e578'}
WORK=Path(tempfile.mkdtemp(prefix='kv-recovery116-'));REPO=WORK/'repo';REPO.mkdir()
HERE=REPO/'experiments/semantic_grounding_method'
DOWNLOAD=WORK/'download';DOWNLOAD.mkdir()
try:
    for relative,expected in ASSETS.items():
        with urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+CORE+'/'+relative,timeout=60) as response:raw=response.read()
        if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed recovery source '+relative)
        target=REPO/relative;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
    os.environ['PYTHONPATH']=str(REPO)
    run_cli([sys.executable,'-u',HERE/'test_kv_json_regression.py'],authenticated=False)
    print('Downloading exact Output116, saved artifacts only.',flush=True)
    run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),
        '--file-pattern',r'(^|/)([^/]+\.jsonl?|[^/]+\.log|checkpoint\.bin)$'])
    candidates=[p.parent for p in DOWNLOAD.rglob('protocol.json')
        if json.loads(p.read_bytes()).get('arm')=='D' and (p.parent/'checkpoint.bin').is_file()]
    if len(candidates)!=1:raise RuntimeError('Output116 lacks a unique D endpoint folder; STOP, no training retry')
    run_cli([sys.executable,'-u',HERE/'recover_kv_summary_cpu.py','--source',str(candidates[0]),
        '--output',str(OUTPUT),'--prepared',str(PREPARED_ROOT),'--version',KERNEL_VERSION,'--code-commit',CORE],authenticated=False)
finally:
    with zipfile.ZipFile(REVIEW,'x',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.glob('*')):
            if path.suffix in ('.json','.jsonl','.log'):archive.write(path,path.name)
        for path in DOWNLOAD.rglob('*.json'):
            if path.name.startswith(('failure_','worker_')):archive.write(path,'download_diagnostics/'+path.name)
        archive.writestr('recovery_execution.json',json.dumps(dict(source_version=KERNEL_VERSION,
            recovery_code=CORE,summary_created=(OUTPUT/'summary.json').is_file(),training_repeated=False,model_loaded=False)))
        if sys.exc_info()[0] is not None:archive.writestr('recovery_error.txt',traceback.format_exc())
    print('SEND RECOVERY ZIP:',REVIEW,flush=True)


Gửi `relation_kv_D_recovered_v116_review.zip`. Giữ Output116/checkpoint gốc.
Bản này không tải raw visual/legacy cache, không chạy GPU, không huấn luyện hoặc chọn checkpoint mới.
